In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.generators import Math2DGenerator
from src.noise import AbsoluteGaussianNoise
from src.shadows import BandShadow
from src.models import RandomForestModel, BaggingPolynomialModel, NeuralNetworkMCDropoutModel
from src.splitter import GapSplitter
from src.orchestrator import RegressionOrchestator
from src.visualization import plot_experiment_results

In [ ]:
exp_functions = [
    ("Tribiala", lambda x: x),
    ("Minimo Global Galera", lambda x: x**2),
    ("Inflexio Puntu Galera", lambda x: x**3),
    ("Deribagarritasun Galera", lambda x: -np.abs(x)),
    ("Jarraitutasun Galera", lambda x: np.sign(x)),
    ("Frekuentzia handiko Zikloen Galera", lambda x: np.sin(3*x))
]

In [ ]:
N_SAMPLES = 500
X_RANGE = (-3, 3)
SHADOW_RANGE = (-1, 1)

noise_strategy = AbsoluteGaussianNoise(std_dev=0.1, seed=42)
shadow = BandShadow(axis='x', min_val=SHADOW_RANGE[0], max_val=SHADOW_RANGE[1])
splitter = GapSplitter(test_size=0.2, random_state=42)

In [ ]:
models = [
    RandomForestModel(n_estimators=100, max_depth=10, random_state=42),
    BaggingPolynomialModel(degree=3, n_estimators=100, random_state=42),
    NeuralNetworkMCDropoutModel(hidden_dim=128, dropout_rate=0.2, epochs=400, n_iter=100)
]

In [ ]:
for nombre_funcion, func_lambda in exp_functions:
    print(f"\n{'='*60}")
    print(f"EBALUAZIOA: {nombre_funcion}")
    print(f"{'='*60}")
    
    generator = Math2DGenerator(function=func_lambda, seed=42)
    
    for modelo in models:
        print(f"\nModel: {modelo.get_name()}")
        
        orchestrator = RegressionOrchestator(generator, splitter, modelo)
        artifacts = orchestrator.run(
            n_samples=N_SAMPLES, 
            x_range=X_RANGE, 
            noise_strategy=noise_strategy, 
            shadow=shadow
        )
        
        if artifacts.uncertainties:
            std_vis = np.mean(artifacts.uncertainties['visible'])
            std_sha = np.mean(artifacts.uncertainties['shadow'])
            print(f"  -> MSE (Itzala): {artifacts.metrics.mse_test_shadow:.4f}")
            print(f"  -> Ezjak. ikusgarrian (std): {std_vis:.4f}")
            print(f"  -> Ezjak. itzalean  (std): {std_sha:.4f}")
        
        titulo_grafica = f"{nombre_funcion}\nEredua: {modelo.get_name()}"
        plot_experiment_results(
            artifacts.split_data, 
            artifacts.predictions, 
            uncertainties=artifacts.uncertainties, 
            title=titulo_grafica
        )